# AI Answer Staleness Flagger — semantic classifier demo

**0 = STABLE · 1 = TIME-SENSITIVE**

This notebook compares a fixed pretrained MiniLM sentence encoder, lexical features and their combination on CPU. It trains a temporal classification head, without encoder fine-tuning or fact verification. Offline execution requires the prepared CSVs, runtime packages and encoder assets.

On Kaggle, attach the project including `models/semantic_encoder/` and select CPU. Statement labels are assistant-authored and need independent human review; these scores are small diagnostic exercises.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent]
if Path('/kaggle/input').exists():
    candidates += [path.parents[2] for path in Path('/kaggle/input').glob('**/data/processed/train.csv')]
ROOT = next((path for path in candidates if (path / 'data/processed/train.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Attach the project containing data/processed/train.csv, or open this notebook from the project folder.')
sys.path.insert(0, str(ROOT))
OUTPUT = Path('/kaggle/working/flagger-output') if Path('/kaggle/working').exists() else ROOT
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Project:', ROOT)
print('Output:', OUTPUT)

## 1. Inspect the data

EverGreenQA questions use the reversed source label mapping. Short public answers carry weak inherited labels. The project adds authored statements with a frozen separate test. Source questions and their answers stay in one group; related authored examples share a family group.

In [ ]:
import csv
from collections import Counter
for split in ['train', 'validation', 'test_questions', 'test_statements']:
    with (ROOT / 'data/processed' / f'{split}.csv').open(encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    print(split, 'n=', len(rows), 'labels=', dict(Counter(row['label'] for row in rows)))

## 2. Sentence splitting and preprocessing

Keep temporal cues, tense, dates and negation. Version numbers and decimal prices must remain intact. This baseline predicts one label per sentence; it does not extract every independent atomic fact.

In [ ]:
from flagger.text import split_claims, normalize_text
sample = 'Python is a programming language. The latest Python version is 3.13. The current price is €50.'
for claim in split_claims(sample):
    print(claim.start, claim.end, normalize_text(claim.text))

## 3. Compare semantic and lexical classifiers

A fixed pretrained encoder supplies sentence vectors. Compare semantic-only and combined heads against the original lexical baseline. A fixed grid uses the existing validation statements to choose settings; neither test set nor the paraphrase diagnostic selects the model. The encoder is not fine-tuned. Provide local encoder assets for offline execution, or allow a one-time pinned download.


In [ ]:
import shutil
from scripts.train_semantic import run
encoder_source = ROOT / 'models/semantic_encoder'
encoder_destination = OUTPUT / 'models/semantic_encoder'
if encoder_source.is_dir() and encoder_source.resolve() != encoder_destination.resolve():
    shutil.copytree(encoder_source, encoder_destination, dirs_exist_ok=True)
report = run(ROOT / 'data/processed', OUTPUT / 'models', OUTPUT / 'reports', offline=encoder_destination.is_dir())


## 4. Report both evaluations

Question performance is not claim performance. The small authored test is an exercise; its labels need independent human review. Confusion matrix rows are assigned labels and columns are predictions, in stable/volatile order.

In [ ]:
print('Selected:', report['selected_model'])
for name, metrics in report['test'].items():
    print(f"{name}: n={metrics['n']}, accuracy={metrics['accuracy']:.3f}, macro-F1={metrics['macro_f1']:.3f}")
    print('Confusion matrix:', metrics['confusion_matrix'])
for kind, metrics in report['paraphrase_diagnostic'].items():
    print('Paraphrases:', kind, metrics['accuracy'])
print('Encoder bytes:', report['metadata']['encoder_bytes'])
print('Encoding seconds:', report['metadata']['encoding_seconds'])
print('Selected head fit seconds:', report['metadata']['head_train_seconds'])


## 5. Predict and highlight an answer

The model reads the input text, its local pretrained encoder and the trained temporal head. A time-sensitive label describes possible change, not falsity. Counts only identify sentences that may need checking.


In [ ]:
from flagger.predict import StalenessClassifier
try:
    from IPython.display import HTML, display
    def show_html(content):
        display(HTML(content))
except ImportError:
    def show_html(content):
        print(content)
from html import escape
classifier = StalenessClassifier(OUTPUT / 'models/staleness.joblib')
result = classifier.analyze(sample)
for claim in result['claims']:
    background, foreground = ('#f1f1f1', '#111') if claim['label'] == 0 else ('#111', '#fff')
    show_html(f"<p style='padding:10px;background:{background};color:{foreground};border-radius:5px'>{escape(claim['text'])} [{claim['label']}]</p>")
print('Stability summary:', result['summary'])
print(f"{result['summary']['volatile']} claims may need current verification.")


## 6. Show that truth is not the target

The false geometry/astronomy statements below describe stable kinds of information. Explicit past prices are historical assertions. None of these are verified by the classifier. These are regression examples, not held-out accuracy measurements.

In [ ]:
examples = ['The Earth is shaped like a cube.', 'An electric current is a flow of electric charge.', 'The visa fee was €50 in 2010.', 'The visa fee is €50.', 'euro will go up', 'The euro went up in 2009.', 'The Sun will rise tomorrow.']
for prediction in classifier.predict_claims(examples):
    print(prediction['label'], prediction['label_name'], prediction['text'])

## 7. Inspect errors and learned features

Keep the frozen test unchanged after reading errors. Improve training data and annotation policy for a subsequent experiment, using a new untouched test set when developing based on test failures.

In [ ]:
import json
with (OUTPUT / 'reports/errors.csv').open(encoding='utf-8') as handle:
    statement_errors = [row for row in csv.DictReader(handle) if row['evaluation_set'] == 'statements']
print('Statement errors:', len(statement_errors))
for row in statement_errors[:5]:
    print('Assigned:', row['gold_label'], 'Predicted:', row['prediction'], row['text'])
features = json.loads((OUTPUT / 'reports/top_features.json').read_text())
print('Stable associations:', [row['feature'] for row in features['stable'][:10]])
print('Volatile associations:', [row['feature'] for row in features['volatile'][:10]])

## Project discussion

Use `docs/bsc_methodology.md`, `docs/semantic_research.md`, `docs/dataset_research.md` and `docs/annotation_guide.md` for the write-up. Review labels with independent annotators and measure agreement. Discuss question-to-statement transfer, weak labels, sentence-level scope, long-sentence truncation and remaining paraphrase errors. Encoder pretraining precedes the measured local head fit. The complete reports and local assets are saved in the output directory.
